# vLLM trên Colab — Setup & Serve

Notebook này clone repo dev từ GitHub, chạy script setup (CUDA/torch/vllm) rồi khởi động vLLM OpenAI-compatible server ở background, cuối cùng gửi thử một request để kiểm tra.

Lưu ý: Colab (T4/L4/A100) chỉ dùng để dev/thử nghiệm nhanh. Môi trường chấm điểm chính thức của cuộc thi là NVIDIA H200, Ubuntu 22.04, CUDA 12.x. Cell **1b** cho chọn 1 trong 3 kịch bản CUDA (dev Colab / nộp BTC / test giống BTC trên Colab) — xem chi tiết trong README. Nếu đổi cấu hình CUDA giữa chừng trong cùng session, cell **2** sẽ tự cài rồi tự restart kernel; sau khi thấy kernel restart, chạy lại từ cell 1b.

In [ ]:
# 1. Clone repo (đổi REPO_URL nếu bạn fork/dùng repo khác)
REPO_URL = "https://github.com/Le-Anh-Duy/vLLM-Colab-Inference.git"
REPO_DIR = "colab-setup"

import os
if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !cd "$REPO_DIR" && git pull

In [ ]:
# 2. Chạy setup (cài CUDA runtime + torch + vllm), tự phát hiện khi cấu hình
# CUDA_VERSION/VLLM_VERSION đổi so với lần chạy trước trong CÙNG session, và
# nếu có đổi thì tự cài xong rồi tự kill kernel để Colab restart (giữ nguyên
# VM/ổ đĩa, chỉ khởi động lại process Python). Sau khi thấy kernel restart,
# CHẠY LẠI CELL NÀY MỘT LẦN NỮA để tiếp tục — lần này sẽ thấy marker khớp và
# bỏ qua cài đặt.
import hashlib
import os
import subprocess
import sys

MARKER_FILE = "/content/.vllm_setup_marker"
tracked_vars = ["CUDA_VERSION", "SKIP_CUDA", "SKIP_PYTHON_DEPS", "VLLM_VERSION"]
signature = hashlib.sha256(
    "|".join(f"{k}={os.environ.get(k, '')}" for k in tracked_vars).encode()
).hexdigest()

previous_signature = None
if os.path.exists(MARKER_FILE):
    with open(MARKER_FILE) as f:
        previous_signature = f.read().strip()

if previous_signature == signature:
    print("==> Cau hinh khong doi so voi lan chay truoc, bo qua setup.")
else:
    changed_mid_session = previous_signature is not None
    ret = subprocess.call(["bash", f"{REPO_DIR}/scripts/setup_vllm.sh"])
    if ret != 0:
        raise RuntimeError(f"setup_vllm.sh that bai voi exit code {ret}")

    with open(MARKER_FILE, "w") as f:
        f.write(signature)

    if changed_mid_session:
        print("==> Cau hinh CUDA vua doi trong session nay, dang restart kernel...")
        print("==> Sau khi kernel restart xong, CHAY LAI TU CELL 1b DE TIEP TUC.")
        os.kill(os.getpid(), 9)

In [ ]:
# 2. Chạy setup (cài CUDA runtime + torch + vllm). Có thể override version qua env var, vd:
# %env TORCH_VERSION=2.11.0
# %env VLLM_VERSION=0.24.0
!bash "{REPO_DIR}/scripts/setup_vllm.sh"

In [ ]:
# 3. Khởi động server ở background, log ra file để không block notebook
import os, subprocess, time

env = os.environ.copy()
env.setdefault("MODEL_NAME", "facebook/opt-125m")
env.setdefault("PORT", "8000")

log_file = open("vllm_server.log", "w")
server_proc = subprocess.Popen(
    ["bash", f"{REPO_DIR}/scripts/serve_vllm.sh"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
)
print(f"server_proc.pid = {server_proc.pid}")

In [ ]:
# 4. Chờ server sẵn sàng (poll /health)
import time, urllib.request, urllib.error

PORT = int(env.get("PORT", "8000"))
url = f"http://localhost:{PORT}/health"

for attempt in range(60):
    try:
        with urllib.request.urlopen(url, timeout=3) as resp:
            if resp.status == 200:
                print("Server san sang.")
                break
    except (urllib.error.URLError, ConnectionError):
        pass
    time.sleep(5)
else:
    print("Server chua san sang, xem log:")
    !tail -n 50 vllm_server.log

In [ ]:
# 5. Test bằng OpenAI client (vllm expose API tương thích OpenAI)
!pip install -q openai

from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="not-needed")
response = client.completions.create(
    model=env["MODEL_NAME"],
    prompt="Xin chao, ban co the gioi thieu ban than khong?",
    max_tokens=64,
)
print(response.choices[0].text)

In [ ]:
# 6. Dừng server khi xong việc
server_proc.terminate()
server_proc.wait()
print("Da dung server.")